# Day 1.2 — Configuring Model Behaviour

In 1.1 we sent one question and accepted whatever text came back. Now we take control of
the two dials the application actually owns:

```text
System instructions + user request  ->  model  ->  better-shaped text
        temperature / max_tokens  ->  how varied and how long
```

By the end you will have seen instructions improve consistency — and seen that they still
do not give you a data structure you can trust.

## Before you begin

### Learning outcomes

- Separate standing instructions (`system`) from the current task (`user`) and see the
  difference in the output.
- Use `temperature` deliberately and explain what value to pick for what job.
- Show that an instruction is not a contract: ask for a dictionary and get prose.

Architecture reference: [D01](../../diagrams/source/day_01.md).

### Expected observation

The constrained answer follows the requested three-section format; the unconstrained one
does not. The "give me a dictionary" answer looks close enough to fool a human and still
breaks `json.loads`.

In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

In [ ]:
# Build the OpenRouter client ONLY when a key exists. Constructing a client (or a
# provider object) unconditionally is the classic way to make a notebook crash on
# the first cell for every student without a key.
COURSE_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b")

client = None
if LIVE:
    from openai import OpenAI          # the OpenAI SDK also speaks to OpenRouter
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",   # this line is what switches provider
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

print("Model to use :", COURSE_MODEL)
print("Client       :", "OpenRouter client ready" if client else "none - using the mock below")

### Step 1 — One `ask` helper for the whole notebook

`ask` builds the request dictionary, sends it only when a key exists, and otherwise
returns a deterministic mock reply. It returns **both** the answer and the payload that
was built, so we can look at the settings we actually sent.

In [ ]:
import json

def mock_reply(messages):
    """Deterministic stand-in for the model.

    It reacts to what was asked, so each demo below shows a genuinely different
    behaviour instead of one canned string.
    """
    system_text = " ".join(m["content"] for m in messages if m["role"] == "system").lower()
    user_text = " ".join(m["content"] for m in messages if m["role"] == "user").lower()

    # (a) The "give me a dictionary" request: helpful prose wrapped around the data.
    if "dictionary" in user_text or "keys" in user_text:
        return (
            "Sure! Here is the dictionary you asked for:\n\n"
            "```python\n"
            "{\n"
            '    "definition": "An AI agent chooses bounded actions using a model.",\n'
            '    "example": "It asks for a calculator tool.",\n'
            '    "limitation": "Host code must execute and authorise the action."\n'
            "}\n"
            "```\n\n"
            "Let me know if you would like me to expand any of the fields!"
        )

    # (b) The temperature demo asks for a short name.
    if "name for" in user_text:
        return "Study Pilot"

    # (c) A system message that asks for the three named sections.
    if "definition" in system_text and "limitation" in system_text:
        return (
            "Definition: An AI agent is an application that uses a model to choose "
            "bounded actions.\n"
            "Example: It requests a calculator tool and the application runs it.\n"
            "Limitation: The model cannot execute anything itself; host code must."
        )

    # (d) No standing instructions: one shapeless paragraph.
    return (
        "An AI agent is a program that uses a language model to work towards a goal, "
        "often over several steps, sometimes using extra capabilities, and people use "
        "the word for many different systems, which is part of why it is confusing."
    )


def ask(messages, max_tokens=400, temperature=0):
    """Return (answer_text, payload_actually_built)."""
    payload = {
        "model": COURSE_MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    if client is None:
        return mock_reply(messages), payload
    try:
        response = client.chat.completions.create(
            **payload, extra_body={"reasoning": {"effort": "low", "exclude": True}}
        )
        return response.choices[0].message.content, payload
    except Exception as exc:
        print("Live call failed, falling back to the mock ->", type(exc).__name__, exc)
        return mock_reply(messages), payload

print("ask() ready. Route:", "OpenRouter" if client else "mock")

### Step 2 — A broad request, with no standing instructions

One `user` message and nothing else. Read the answer and ask yourself: could a program
rely on this shape?

In [ ]:
broad_answer, _ = ask([{"role": "user", "content": "Explain an AI agent."}])
print("--- answer to a bare user message ---")
print(broad_answer)
print()
print("Lines in the answer :", len(broad_answer.splitlines()))
print("Named sections      :", sum(word in broad_answer for word in ("Definition", "Example", "Limitation")), "of 3")

### Step 3 — Add a system message

A **system** message carries standing instructions: how to behave for this whole call. A
**user** message carries the current task. Keeping them apart means you can change the
task without rewriting the rules, and vice versa.

In [ ]:
constrained_messages = [
    {
        "role": "system",
        "content": (
            "You teach engineering students who are new to agentic AI. Use plain language. "
            "Answer in exactly three short sections labelled Definition, Example, and "
            "Limitation, each one sentence long."
        ),
    },
    {"role": "user", "content": "Explain an AI agent."},
]

constrained_answer, _ = ask(constrained_messages)
print("--- answer with standing instructions ---")
print(constrained_answer)
print()
print("Named sections      :", sum(word in constrained_answer for word in ("Definition", "Example", "Limitation")), "of 3")

### Step 4 — `temperature`: how much the model is allowed to vary

`temperature` scales the randomness of the next-token choice.

- `0` — always take the most likely token. Best for extraction, classification, tool use
  and anything you want to be reproducible. Every notebook in this course uses `0`.
- `1` — sample more freely. Useful for brainstorming, bad for pipelines.

Watch the value we actually put in the request, and then what comes back.

In [ ]:
question = [{"role": "user", "content": "Give one two-word name for a study-planner agent."}]

cold_answer, cold_payload = ask(question, max_tokens=60, temperature=0)
hot_answer, hot_payload = ask(question, max_tokens=60, temperature=1)

print("temperature field sent (cold):", cold_payload["temperature"])
print("temperature field sent (hot) :", hot_payload["temperature"])
print()
print("temperature=0 ->", cold_answer.strip()[:120])
print("temperature=1 ->", hot_answer.strip()[:120])
print()
if client is None:
    print("MOCK mode: the mock ignores temperature entirely, so both answers are identical.")
    print("That is exactly the point - the setting is a request to the PROVIDER. The mock")
    print("still shows you the field is present and correct in the payload we built.")
else:
    print("LIVE mode: run this cell a few times. At 0 the answer barely moves; at 1 it")
    print("wanders. Same prompt, same model - only the sampling rule changed.")

### Step 5 — Break it: ask for a data structure

Instructions shape *style* well. Now ask for something a program must parse and try to use
the answer directly.

In [ ]:
dict_answer, _ = ask([{"role": "user", "content": (
    "Explain an AI agent as a dictionary with exactly the keys definition, example, "
    "and limitation."
)}])

print("--- what the model returned ---")
print(dict_answer)
print()
print("--- what happens when the application tries to use it ---")
try:
    parsed = json.loads(dict_answer)
    print("Parsed successfully:", parsed)
except json.JSONDecodeError as error:
    print("json.loads FAILED:", error)
    print()
    print("The data is in there, but it is surrounded by a greeting, a Markdown fence and")
    print("a follow-up offer. A human reads past all that; json.loads cannot.")

### Try it yourself

Could you fix this by stripping the Markdown fence with `str.replace`? Predict whether
that is a reliable fix, then run the worked solution.

In [ ]:
# --- Worked solution ---
# A "clean it up afterwards" fix works on the example in front of you and fails on the
# next one. Below we strip the fence and see it parse - then we feed it a slightly
# different, equally plausible reply and watch the same code break.

def strip_fence(text):
    """Remove a leading prose paragraph and a ```python / ``` fence, if present."""
    if "```" not in text:
        return text
    inner = text.split("```")[1]                # take what is between the first two fences
    return inner.replace("python", "", 1).strip()

cleaned = strip_fence(dict_answer)
try:
    print("Cleaned and parsed:", json.loads(cleaned))
except json.JSONDecodeError as error:
    print("Even after cleaning:", error)

# The next equally reasonable reply uses single quotes and a trailing comment.
other_reply = "Here you go:\n\n{'definition': 'x', 'example': 'y', 'limitation': 'z'}  # done"
try:
    print("Second reply parsed:", json.loads(strip_fence(other_reply)))
except json.JSONDecodeError as error:
    print("Second reply FAILED:", error)
    print()
    print("Conclusion: cleaning up text is guesswork. In 1.3 we stop guessing and make the")
    print("provider return JSON that matches a schema, then validate it with Pydantic.")

### Checkpoint

**1. You want a classifier that labels support tickets as `bug`, `question` or `praise`. What temperature do you choose, and where do the label rules belong — system or user message?**

<details><summary>Show answer</summary>

`temperature=0`, because you want the same ticket to get the same label every time and you
are not looking for creativity. The label definitions are standing instructions that apply
to every ticket, so they belong in the **system** message; the ticket text itself is the
**user** message. That split lets you send thousands of tickets without rewriting the rules.

</details>

**2. The three-section instruction worked. Why is that still not enough for a program that needs `definition`, `example` and `limitation` as fields?**

<details><summary>Show answer</summary>

Because "it worked" is an observation about one run, not a guarantee. The model may add a
friendly opening line, rename a heading, merge two sections, or answer in a different
order — all of which a human forgives and `json.loads` does not. An instruction is a
preference; a schema plus validation is a contract, which is what 1.3 adds.

</details>

### Recap

- **Limitation we saw:** a bare user message produces shapeless prose, and even a good
  system message cannot promise a parsable structure.
- **Layer we added:** deliberate configuration — a system/user split, `max_tokens`, and
  `temperature=0` for reproducible work.
- **Evidence it worked:** the constrained answer contained all three named sections while
  the bare one contained none, and `json.loads` still failed on the "dictionary" reply.